# Tutorial 4: DOE Planning (`grid` vs `sobol`)

Estimated time: 20-35 minutes

## Prerequisites
No optional dependencies required.

## Learning aims
- Primary package aim: compare DOE strategies and understand planning outputs
- Secondary scientific aim: reason about coverage, budget, and bias in simulation campaigns

## Success criteria
- you can justify a DOE strategy for a given model dimensionality and run budget


## Why this tutorial matters
Each step connects the CLI workflow to scientific reasoning so you can explain not only *what* ran, but *why* results are meaningful.


## Step 1: Run both planning commands


In [ ]:
# Cross-platform setup — works on Windows / macOS / Linux.
# Locates the repo root, puts src/ on sys.path, and defines helpers that run
# the `mm` CLI in-process (run_mm_cli) and auxiliary tools like pytest/ruff
# via the active interpreter (run_tool). No shell cells, no PYTHONPATH prefix.
import io
import os
import subprocess
import sys
from contextlib import contextmanager, redirect_stderr, redirect_stdout
from pathlib import Path

root = Path.cwd().resolve()
if not (root / "src").is_dir() and (root.parent / "src").is_dir():
    root = root.parent
if not (root / "src").is_dir():
    raise RuntimeError("Could not locate project root (expected src/).")

src_path = root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Run the whole notebook from the repo root so CLI artifacts and any
# CWD-relative registry lookups (e.g. eval_surrogate) resolve consistently.
os.chdir(root)

# Jupyter caches imported modules; clear bayesian_metamodeling so re-runs pick
# up the current local source.
for module_name in list(sys.modules):
    if module_name == "bayesian_metamodeling" or module_name.startswith("bayesian_metamodeling."):
        del sys.modules[module_name]

from bayesian_metamodeling.cli.main import main as mm_main


@contextmanager
def in_project_root():
    previous = Path.cwd()
    os.chdir(root)
    try:
        yield
    finally:
        os.chdir(previous)


def run_mm_cli(*args: str, check: bool = True) -> int:
    """Run `mm <args>` in-process; cross-platform, no shell."""
    stdout_buf = io.StringIO()
    stderr_buf = io.StringIO()
    previous_argv = sys.argv[:]
    try:
        sys.argv = ["mm", *args]
        with in_project_root(), redirect_stdout(stdout_buf), redirect_stderr(stderr_buf):
            exit_code = mm_main()
    finally:
        sys.argv = previous_argv

    print("$ mm", " ".join(args))
    out = stdout_buf.getvalue().strip()
    err = stderr_buf.getvalue().strip()
    if out:
        print(out)
    if err:
        print(err)
    if check and exit_code != 0:
        raise RuntimeError(f"CLI command failed ({exit_code}): mm {' '.join(args)}")
    return exit_code


def run_tool(*args: str, check: bool = True) -> int:
    """Run an auxiliary tool (pytest, ruff) via the active interpreter, cross-platform."""
    cmd = list(args)
    if cmd and cmd[0] in {"pytest", "ruff"}:
        cmd = [sys.executable, "-m", *cmd]
    print("$", " ".join(args))
    with in_project_root():
        result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f"Tool failed ({result.returncode}): {' '.join(args)}")
    return result.returncode


In [ ]:
run_mm_cli('plan', 'tutorials/specs/model.toy.grid.json')
run_mm_cli('plan', 'tutorials/specs/model.toy.sobol.json')


## Step 2: Plot DOE points (graphic)


In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

root = Path.cwd().resolve()
if not (root / "src").is_dir() and (root.parent / "src").is_dir():
    root = root.parent

grid_spec = json.loads((root / 'tutorials/specs/model.toy.grid.json').read_text())
sobol_spec = json.loads((root / 'tutorials/specs/model.toy.sobol.json').read_text())

# grid points
a_vals = grid_spec['design']['grid']['a']
b_vals = grid_spec['design']['grid']['b']
grid_pts = np.array([(a, b) for a in a_vals for b in b_vals], dtype=float)

# approximate sobol points from existing runs if available
runs_root = root / 'tmp/tutorials/toy_store_sobol/runs'
sobol_pts = []
if runs_root.exists():
    for run_dir in sorted(p for p in runs_root.iterdir() if p.is_dir()):
        inp = json.loads((run_dir / 'inputs.json').read_text())
        sobol_pts.append((inp['a'], inp['b']))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.scatter(grid_pts[:, 0], grid_pts[:, 1], s=90)
plt.title('Grid DOE points')
plt.xlabel('a')
plt.ylabel('b')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
if sobol_pts:
    sobol_arr = np.asarray(sobol_pts, dtype=float)
    plt.scatter(sobol_arr[:, 0], sobol_arr[:, 1], c='tab:orange', s=90)
    plt.title('Sobol DOE points (from runs)')
else:
    plt.text(0.1, 0.5, 'No Sobol runs yet\nRun `mm run tutorials/specs/model.toy.sobol.json`', fontsize=10)
    plt.title('Sobol DOE points')
plt.xlabel('a')
plt.ylabel('b')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Scientific checkpoint
Write a short decision note:
- when grid is better,
- when sobol is better,
- which one you would pick for a 6-10D study and why.
